# Reproducing Martinez-Conde et al. (2019) from CRCNS pvc-12

**Paper.** Martinez-Conde S, McCamy MB, Troncoso XG, Otero-Millan J, Macknik SL (2019). Area V1
responses to illusory corner-folds in Vasarely's nested squares and the Alternating Brightness
Star illusions. *PLoS ONE* 14(3): e0210941.
[doi:10.1371/journal.pone.0210941](https://doi.org/10.1371/journal.pone.0210941). The claims file
was written from the version of record (PMC6438452, Europe PMC full text), without the
supplementary files S1 and S2 Table.

**Dataset.** CRCNS [pvc-12](https://crcns.org/data-sets/pvc/pvc-12),
[doi:10.6080/K05D8Q1T](https://doi.org/10.6080/K05D8Q1T): one zip, `V1_Vasarely_data.zip`,
streamed from the public bucket `crcnsarchive` (`pvc-12/live/`) and opened in place. CRCNS data
sets have no versions, so the version is recorded as `unversioned`, and the MD5 of the zip from
the data set's `checksums.md5` (e1301fe463e72332e2303b2053ee0559) is printed below so that a later
change can be detected. The paper cites the data set by its DOI only; there is one published copy
(`live/`), and it is the one used.

**Data descriptor.** The data set's documentation (version 0.7, Feb 15, 2018) describes per
neuron and experiment a spike rate map file and a stimulus file, and says the data set holds 53
neurons. The files hold an ROI-registered copy of each map (`Standard`) that the descriptor does
not mention; the analysis rests on it (see the data check).

| Claim | Tested here |
|---|---|
| neuron-counts | tested |
| spikes-per-condition | blocked: the spike counts and eye times are not in the data set |
| s4-baseline-no-difference | tested |
| fig4a-above-baseline | tested |
| fig4b-spike-rate-difference | tested |
| fig4b-pairwise-significance | tested |
| fig5-average-maps | tested |
| table1-edge, table1-p180, table1-s90, table1-s52, table1-s40 | tested |
| fig6-midpoint-representative | tested |
| fig7a-rms-contrast-fit | tested (from the stimulus images) |
| fig7c-fold-edge-equivalence | tested |
| fig7c-fold-edge-significance | tested |
| fig7d-nonlinear | tested |
| fig7d-dog-similar | blocked: the DOG model output of reference [3] is not public as data |
| s3-latency-peak | blocked: the maps are made at one latency (60 ms); no spike times or eye positions |

In [ ]:
import time
import resource
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import least_squares
from scipy.signal import fftconvolve
from statsmodels.stats.multitest import multipletests

import vasarely as v

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
print("checksums.md5 of the data set:")
print(v.checksums())
z = v.open_zip()
recs = v.load_all(z)
sheet = v.spreadsheet(z)
print(f"data set {v.DATASET_ID} ({v.DATASET_VERSION}, doi:{v.DOI}); {len(recs)} spike rate map files "
      f"loaded in {time.time() - started:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{k: r.get(k) for k in ["file", "condition", "prefix", "cell", "suffix", "monkey", "format",
                                             "registered"]} | {
    "has_BaseActivity": bool(np.isfinite(r.get("base_dataset", np.nan))),
    "fields": ",".join(r["fields"]) if isinstance(r["fields"], list) else r["fields"]} for r in recs])
print("Spike rate map files by experiment, format, and whether they hold the registered map `Standard`:")
print(pd.crosstab([info.condition, info.format], info.registered, margins=True).to_string())
print("\n... and the undocumented BaseActivity:")
print(pd.crosstab(info.condition, info.has_BaseActivity).to_string())
print("\nFiles by monkey (J or A, from the cell code) and experiment:")
print(pd.crosstab(info.monkey, info.condition, margins=True).to_string())
print("\nTop-level fields of the spike rate map files (count of files):")
print(info.fields.value_counts().to_string())
print("\nEvery stimulus file has a spike rate map file and the reverse:",
      sorted(n.replace("_Stimulus", "") for n in z.namelist() if n.endswith("_Stimulus.mat")) ==
      sorted(n.replace("_SpikeRateMap", "") for n in z.namelist() if n.endswith("_SpikeRateMap.mat")))

The descriptor names three fields (image, smoothed_image, smoothed_image_norm); every file has
the first two, 38 of 77 have `smoothed_image_norm`, and 60 of 77 have `Standard`, the map
registered to the stimulus. All 17 files without `Standard` are Star files, two of them (prefix
ABS) in an older layout (one saved as MATLAB v7.3). Prefixes are ABC (Corners), ABD and ABS (Star)
and AEC (Edge-compare); the letter after the cell number (A to E) is not explained anywhere.

In [ ]:
cells_files = info.groupby("cell").condition.apply(lambda s: sorted(s)).rename("files")
sheet_marks = sheet.set_index("Cell")[["Corners", "Star", "Edge-Compare"]]
sheet_marks.columns = ["Corners", "Star", "Edge-compare"]
listed = sheet_marks.apply(lambda r: sorted(c for c in r.index if r[c]), axis=1).rename("spreadsheet")
cmp = pd.concat([listed, cells_files], axis=1)
cmp["files"] = cmp["files"].apply(lambda x: x if isinstance(x, list) else [])
cmp["spreadsheet"] = cmp["spreadsheet"].apply(lambda x: x if isinstance(x, list) else [])
print(f"Spreadsheet rows (cells): {len(sheet)}; with at least one experiment marked: {int(sheet_marks.any(axis=1).sum())}")
print("Spreadsheet marks per experiment:", sheet_marks.sum().to_dict())
print("Cells with files:", cells_files.size)
mismatch = cmp[cmp.spreadsheet.apply(tuple) != cmp.files.apply(tuple)]
print("\nCells where the spreadsheet and the files disagree:")
print(mismatch.to_string() if len(mismatch) else "none")
print("\nSpreadsheet rows with no experiment marked:", sheet.loc[~sheet_marks.any(axis=1).values, "Cell"].tolist())

The spreadsheet lists 58 cells, the paper's number of analysed neurons, but marks an experiment for
only 53, which is the number the data descriptor gives; the files match the spreadsheet's marks
cell by cell.

In [ ]:
# Duplicates: identical maps deposited under two names
import hashlib
hashes = {}
for r in recs:
    if r["registered"]:
        h = hashlib.md5(np.round(r["rate"], 9).tobytes()).hexdigest()
        hashes.setdefault(h, []).append(r["file"])
dups = [f for f in hashes.values() if len(f) > 1]
print("Registered maps that are identical in two files:", dups if dups else "none")
# Also the gaze-frame maps of all v5 files
full = {}
for name in sorted(n for n in z.namelist() if n.endswith("_SpikeRateMap.mat")):
    d = v.read_mat(z, name)
    if hasattr(d, "_fieldnames"):
        full.setdefault(hashlib.md5(np.asarray(d.smoothed_image).tobytes()).hexdigest(), []).append(name.split("/")[-1])
print("Gaze-frame maps that are identical in two files:", [f for f in full.values() if len(f) > 1] or "none")

### Units and smoothing of the maps

In [ ]:
eye = [r for r in recs if "eyetime_s" in r]
for r in eye:
    print(f"{r['file']}: eye time {r['eyetime_s']:.1f} s; image * eyetime is an integer count at every pixel: "
          f"{r['count_is_integer']}; total {r['spikes_from_eyetime']:.0f} spikes")
mx = pd.Series({r["file"]: r["full_image_max"] for r in recs if "full_image_max" in r})
print(f"Maximum of `image` over files: min {mx.min():.3f}, max {mx.max():.3f} (a pixel seen for one 1 ms sample with one spike)")
sm = pd.Series({r["file"]: r["full_smoothed_max_hz"] for r in recs if "full_smoothed_max_hz" in r})
print(f"Peak of smoothed_image times 1000 (spikes/s): {sm.min():.1f} to {sm.max():.1f}")
reg = [r for r in recs if r["registered"]]
rate_all = np.concatenate([r["rate"].ravel() for r in reg])
print(f"Registered smoothed maps (spikes/s): min {rate_all.min():.2f}, median {np.median(rate_all):.2f}, max {rate_all.max():.1f}")
print("Registered map sizes (rows x cols) by experiment:",
      {c: sorted({r['rate'].shape for r in reg if r['condition'] == c}) for c in v.CONDITIONS})

`image` is the unsmoothed spike count divided by the eye time in 1 ms samples: in the one file
that also stores `eyetime`, the product is an integer count everywhere. So the maps are spikes per
sample, and times 1000 they are spikes/s, consistent with the paper's units (Table 1, spikes per s)
and with peak rates of 18 to 140 spikes/s. The descriptor calls `image` "the raw spikes counts";
the values are rates, not counts.

In [ ]:
# The undocumented `norm` and the descriptor's `smoothed_image_norm`: do they hold eye time or counts?
nv = pd.Series({r["file"]: str(r["norm_values"]) for r in recs if "norm_values" in r})
print(f"`norm` in {len(nv)} files; its values: {nv.value_counts().to_dict()}")
sn = pd.DataFrame([{"file": r["file"], "condition": r["condition"], "relation": r["smoothed_norm_relation"],
                    "min": r["smoothed_norm_range"][0], "max": r["smoothed_norm_range"][1]}
                   for r in recs if "smoothed_norm_relation" in r])
print(f"\n`smoothed_image_norm` in {len(sn)} files; how it relates to smoothed_image (sm) and BaseActivity (b), per 1 ms sample:")
print("  difference = sm - b; index = (sm - b) / (sm + b); other = neither")
print(sn.groupby("relation").agg(files=("file", "size"), min=("min", "min"), max=("max", "max")).round(4).to_string())
print("Largest eye time at a pixel (1 ms samples) in the file that stores it:", [r["eyetime_max"] for r in recs if "eyetime_max" in r])
print("Files with relation 'other':", sn.loc[sn.relation.str.startswith("other"), "file"].tolist())


`norm` is the scalar 0 in all 74 files that have it (73 MATLAB v5 files and the v7.3 file). `smoothed_image_norm` (38 files) is the
smoothed map normalised by the file's `BaseActivity`: the difference sm - b in 5 files and the
index (sm - b) / (sm + b) in 12; in the other 21 it is neither, with values from -1.02 to 2.92. In
no file is it an eye time or a count: eye time in 1 ms samples reaches 1504 at a pixel in the one
file that stores it, and these values stay within -1.02 to 2.92. So neither field gives spike
totals, and spikes-per-condition stays blocked. That `BaseActivity` is used to normalise the maps
suggests it is the authors' own baseline, though the files do not say how it was computed.

In [ ]:
def fspecial(n, s):
    x = np.arange(n) - (n - 1) / 2
    g = np.exp(-(x[:, None] ** 2 + x[None, :] ** 2) / (2 * s * s))
    return g / g.sum()

rows = []
for name in ["Edge-compare/AEC071800J1", "Star/ABD071200J1", "Corners/ABC050900J1A"]:
    d = v.read_mat(z, f"V1_Vasarely_data/{name}_SpikeRateMap.mat")
    img = d.image.toarray()
    for n, s in [(16, 7), (16, 5)]:
        c = fftconvolve(img, fspecial(n, s), mode="full")[n // 2:n // 2 + 480, n // 2:n // 2 + 640]
        rows.append({"file": name, "kernel": f"{n} px, SD {s}", "r": np.corrcoef(c.ravel(), d.smoothed_image.ravel())[0, 1],
                     "max abs diff / peak": np.abs(c - d.smoothed_image).max() / d.smoothed_image.max()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

Smoothing `image` with the Gaussian of the Fig 3B legend (16 DU, SD 7 DU) does not give
`smoothed_image`: the two differ by 19% to 27% of the peak. A 16 DU Gaussian with SD 5 DU comes
much closer (r = 0.999, 5% to 7% of the peak) but is not exact either. The analysis uses the
files' own smoothed maps.

### Registration of the maps to the stimulus

`Standard` holds the map rotated so that the aligned fold is vertical and cropped to the ROI, with
the stimulus's equal-luminance polygons in the same frame. The map in `Standard` matches the
gaze-frame `smoothed_image` after the same rotation as the stimulus and a shift that differs from
neuron to neuron (the receptive field position), which the files do not store. Below, for each
fold, the stimulus luminance read from each file's own stimulus image, resampled into the
Standard frame by the transform fitted on the polygons, at 5%, 50% and 95% of the fold, and the
"ridge" (luminance on the fold minus beside it; positive on a bright fold).

In [ ]:
chk = pd.DataFrame([{"file": r["file"], "condition": r["condition"], **v.check_luminance(r)} for r in reg])
for c, g in chk.groupby("condition"):
    g = g.dropna(axis=1, how="all").drop(columns=["file", "condition"])
    print(f"--- {c} ({len(g)} files): luminance, % (min to max over files)")
    print(g.agg(["min", "max"]).T.round(1).to_string())
print("\nBright S40 fold per Corners file:", pd.Series([v.bright_corner_fold(r) for r in reg if r["condition"] == "Corners"]).value_counts().to_dict())

In [ ]:
sh = pd.DataFrame([{"file": r["file"], "cell": r["cell"], "condition": r["condition"], "shift_x": r["gaze_shift"][0],
                    "shift_y": r["gaze_shift"][1], "r": r["gaze_shift_r"]} for r in reg])
print(f"Standard map against the rotated, shifted gaze-frame map: r from {sh.r.min():.3f} to {sh.r.max():.3f} over {len(sh)} files")
multi = sh[sh.cell.isin(sh.cell[sh.cell.duplicated()])].sort_values(["cell", "condition"])
print("Shift (DU, display frame) for cells registered in more than one experiment:")
print(multi.round(1).to_string(index=False))
spread = multi.groupby("cell").apply(lambda g: np.hypot(g.shift_x.max() - g.shift_x.min(), g.shift_y.max() - g.shift_y.min()))
print(f"Largest distance between the shifts of one cell, per cell: {spread.round(1).to_dict()}")

Every `Standard` map is the gaze-frame `smoothed_image` rotated like the stimulus and shifted
(r of at least 0.996). The shift stands for the receptive field position, which the files do not
store, and it is not the same for one cell across experiments (distances printed above), so it
cannot be taken from another experiment of the same cell. The 17 Star maps without `Standard`
therefore cannot be placed on the stimulus here.

In every registered file the Square, Star, P180 and Corners folds run from 0% to 100% luminance as
the paper describes (the printed points at 5%, 50% and 95% lie within 6 points of these), the
Square and Star folds are bright ridges, P180 is flat (ridge 0), and the baseline squares lie on
0% luminance (the Corners images code 0% as 1.01). Of the two 40 degree folds of Corners, the one
whose apex points down in the Standard frame is the bright one in all 26 files; it is the S40 fold
used. The Edge is the boundary of the rectangle, so a 2 DU check box on it is partly off the
rectangle, and its luminance is lower than the level (Edge@50% from 24% to 50%, Edge@95% from 47%
to 95%); this check does not show the Edge's levels, which are read from the rectangle's polygons.
The Edge "ridge" is not meaningful for the same reason.

A spatial check that the maps are registered to the stimulus: on Square and Edge, the paper's
neurons respond along the Edge (Table 1). The cell below compares, per neuron, the mean rate in a
band 8 DU wide along the Edge with the same band moved sideways by 24 to 64 DU, in both directions.

In [ ]:
rows = []
for r in reg:
    if r["condition"] != "Edge-compare":
        continue
    M = r["rate"]
    x, y0, y1 = r["folds"]["Edge"]
    yc, L = (y0 + y1) / 2, abs(y1 - y0)
    on = v.box_mean(M, x, yc, 8, L)
    off = [v.box_mean(M, x + s, yc, 8, L) for s in (-64, -48, -32, -24, 24, 32, 48, 64)]
    rows.append({"file": r["file"], "on_edge": on, "max_shifted": np.nanmax(off), "on_is_max": on > np.nanmax(off)})
reg_chk = pd.DataFrame(rows)
print(reg_chk.round(2).to_string(index=False))
print(f"\nEdge band above every shifted band in {reg_chk.on_is_max.sum()} of {len(reg_chk)} neurons; "
      f"median on {reg_chk.on_edge.median():.1f} against median of the shifted maxima {reg_chk.max_shifted.median():.1f} spikes/s")

The band along the Edge is above every sideways shift in 16 of 22 neurons (median 21.7 against
17.2 spikes/s). This is consistent with the registered maps lying on the stimulus as the polygons
say; it does not prove the registration of each neuron, which rests on the authors' `Standard`.

In [ ]:
# Are the 12 registered Star neurons like the 16 Star files without `Standard`? Gaze-frame maps only.
st = pd.DataFrame([{"file": r["file"], "registered": r["registered"], "peak": r["full_smoothed_max_hz"],
                    "median": r["full_smoothed_median_hz"], "monkey": r["monkey"]}
                   for r in recs if r["condition"] == "Star" and "full_smoothed_max_hz" in r])
print("Star files read (v5):", st.registered.value_counts().to_dict(), "; by monkey:",
      st.groupby("registered").monkey.apply(lambda m: m.value_counts().to_dict()).to_dict())
print(st.groupby("registered")[["peak", "median"]].describe().round(2).T.to_string())
for col in ["peak", "median"]:
    a, u = st.loc[st.registered, col], st.loc[~st.registered, col]
    print(f"{col} rate of the gaze-frame smoothed map, registered vs not: medians {a.median():.2f} vs {u.median():.2f} spikes/s; "
          f"rank sum p = {stats.ranksums(a, u).pvalue:.3f}")


The 16 Star maps that cannot be registered (all from monkey J) do not differ detectably from the 12
that can in the peak (medians 41.59 against 52.14 spikes/s, rank sum p = 0.49) or the median rate
of the gaze-frame map (6.59 against 8.80, p = 0.75). The median over the visited pixels is a rough
stand-in for the baseline, which cannot be placed without the registration; the comparison does
not show that the two groups have the same fold responses. The ABS file saved as MATLAB v7.3 is
not in the comparison.

### Baseline

Readings of the absolute baseline carried through (see the helper's docstring): "squares", the
16 x 16 DU squares the Methods describe, inside the corners of the circumscribing square (the
main reading); "centred", the same squares centred on those corners; and "dataset", the
undocumented `BaseActivity` of the file where there is one. The Corners placement is an
assumption in both square readings. The cell compares the squares with `BaseActivity`; the cell
after it adds two more readings, "nearest" and "corners_quarter".

In [ ]:
BASE_COL = {"squares": "base_squares", "centred": "base_centred", "dataset": "base_dataset"}
base_rows = []
for r in reg:
    centres = v.baseline_squares(r["rate"].shape, r["condition"], r["folds"])
    centred = v.baseline_squares(r["rate"].shape, r["condition"], r["folds"], placement="centred")
    cover = [v.box_mask(r["rate"].shape, x, y, v.BASE_SIZE, v.BASE_SIZE).sum() / v.BASE_SIZE ** 2 for x, y in centred]
    base_rows.append({"file": r["file"], "cell": r["cell"], "monkey": r["monkey"], "condition": r["condition"],
                      "base_squares": v.baseline(r["rate"], centres), "base_centred": v.baseline(r["rate"], centred),
                      "base_dataset": r["base_dataset"],
                      "base_squares_unsmoothed": v.baseline(r["rate_unsmoothed"], centres),
                      "centred_cover_min": min(cover)})
B = pd.DataFrame(base_rows)
both = B.dropna(subset=["base_dataset"])
print(f"Files with both: {len(both)}; Pearson r = {np.corrcoef(both.base_squares, both.base_dataset)[0, 1]:.3f}; "
      f"median |difference| {np.median(np.abs(both.base_squares - both.base_dataset)):.2f} spikes/s")
print(both.groupby("condition").apply(lambda g: pd.Series({
    "n": len(g), "r": np.corrcoef(g.base_squares, g.base_dataset)[0, 1],
    "median squares": g.base_squares.median(), "median dataset": g.base_dataset.median()})).round(2).to_string())
print("\nBaseline range (spikes/s), squares:", B.base_squares.min().round(2), "to", B.base_squares.max().round(2),
      "; dataset:", B.base_dataset.min().round(2), "to", B.base_dataset.max().round(2))
print("Files whose BaseActivity is exactly 0:", B.loc[B.base_dataset == 0, "file"].tolist())
print("\nCentred squares (variant): smallest fraction of a square inside the ROI, by experiment:",
      B[B.condition != "Corners"].groupby("condition").centred_cover_min.min().round(2).to_dict(),
      "(Corners keeps its one placement)")
print("Largest luminance (%) under the centred squares, inside the ROI:",
      max(np.nanmax(r["lum"][v.box_mask(r["lum"].shape, x, y, v.BASE_SIZE, v.BASE_SIZE)])
          for r in reg if r["condition"] != "Corners"
          for x, y in v.baseline_squares(r["lum"].shape, r["condition"], r["folds"], placement="centred")))
print("\nPer file, baseline (spikes/s) under the three readings:")
print(B[["file", "condition", "base_squares", "base_centred", "base_dataset"]].round(2).to_string(index=False))


The two baselines agree only roughly (r and median difference printed above); how `BaseActivity`
was computed is not documented. Two files have a `BaseActivity` of exactly 0 spikes/s. The
Methods put the Square and Edge squares "at the corners of the 8.84 x 8.84 DVA (138 x 138 DU)
circumscribing square" (Methods, Square and edge) and the Star squares "at the corners of the
circumscribing square" (Methods, Star), which allows squares inside the corners (reading
"squares") or centred on them (reading "centred"). The centred squares lie wholly on 0%
luminance (largest luminance under them 0.0, printed above); for Square and Edge up to 19% of a
square falls outside the ROI (smallest fraction inside 0.81) and that part is dropped. For
Corners the two readings are the same.

In [ ]:
# Which placement of the squares agrees with the files' own BaseActivity? Per stimulus, on the files that have both
agree = []
for c, gg in B.dropna(subset=["base_dataset"]).groupby("condition"):
    di = (gg.base_squares - gg.base_dataset).abs()
    dc = (gg.base_centred - gg.base_dataset).abs()
    p = stats.wilcoxon(di, dc).pvalue if (di != dc).any() else np.nan
    agree.append({"condition": c, "n": len(gg), "inside closer": int((di < dc).sum()), "centred closer": int((dc < di).sum()),
                  "median |inside - BaseActivity|": di.median(), "median |centred - BaseActivity|": dc.median(),
                  "mean inside - BaseActivity": (gg.base_squares - gg.base_dataset).mean(),
                  "mean centred - BaseActivity": (gg.base_centred - gg.base_dataset).mean(), "signed rank p": p})
agree = pd.DataFrame(agree).set_index("condition")
print(agree.round(3).to_string())
nearest = {c: ("base_centred" if row["median |centred - BaseActivity|"] < row["median |inside - BaseActivity|"] else "base_squares")
           for c, row in agree.iterrows()}
print("Placement nearer BaseActivity, per stimulus:", nearest)

# Corners variant (the text does not place its two squares): the two squares at a quarter and three quarters of
# the ROI's width on its bottom edge, also on 0% luminance
for r, b in zip(reg, base_rows):
    b["base_nearest"] = b[nearest[r["condition"]]]
    if r["condition"] == "Corners":
        H, W = r["rate"].shape
        hs = v.BASE_SIZE / 2
        b["base_corners_quarter"] = v.baseline(r["rate"], [(W / 4, H - hs + 0.5), (3 * W / 4, H - hs + 0.5)])
        b["corners_quarter_max_lum"] = max(np.nanmax(r["lum"][v.box_mask(r["lum"].shape, x, H - hs + 0.5, v.BASE_SIZE, v.BASE_SIZE)])
                                           for x in (W / 4, 3 * W / 4))
    else:
        b["base_corners_quarter"] = b["base_squares"]
B = pd.DataFrame(base_rows)
BASE_COL.update(nearest="base_nearest", corners_quarter="base_corners_quarter")
C = B[B.condition == "Corners"]
print(f"\nCorners, {len(C)} files: largest luminance under the quarter squares {C.corners_quarter_max_lum.max():.2f} (0% is coded 1.01)")
print(C[["base_squares", "base_corners_quarter", "base_dataset"]].describe().loc[["count", "mean", "50%", "min", "max"]].round(2).to_string())
for col in ["base_squares", "base_corners_quarter"]:
    cc = C.dropna(subset=["base_dataset"])
    print(f"{col} against BaseActivity ({len(cc)} files): median |difference| {np.median(np.abs(cc[col] - cc.base_dataset)):.2f}, "
          f"r = {np.corrcoef(cc[col], cc.base_dataset)[0, 1]:.3f}")


For Square and Edge, the centred squares are nearer the files' `BaseActivity` than the squares
inside the corners: in 7 of 9 files, with a median absolute difference of 0.67 against 1.51
spikes/s (signed rank p = 0.027) and a mean difference of 0.003 against 0.86. For Star it is the
reverse: the squares inside the corners are nearer in 8 of 10 files (0.47 against 1.37, p =
0.020). For Corners, the two bottom corners of the ROI are nearer `BaseActivity` (median absolute
difference 0.72, r = 0.868) than two squares at a quarter and three quarters of the bottom edge
of the ROI (2.88, r = 0.895), which also lie on 0% luminance. `BaseActivity` itself is not the
baseline the Methods describe, but it is the authors' own and close to the squares (r = 0.919),
so it is the best evidence the files give on where the squares were.

Reading "nearest" takes, per stimulus, the placement nearer `BaseActivity`: centred for Square
and Edge, inside the corners for Star, the bottom corners for Corners. Reading
"corners_quarter" is the main reading with the Corners squares moved to the quarter points.
Both are carried through every claim in the section "Baseline readings compared", before the
summary.

### Per-neuron measures

All claims below use these tables. Each registered file gives, per fold, the midpoint response
(8 x 8 DU), the spike rate difference (response minus baseline) and the ten block differences.
Neurons are those with a registered map: 22 Square and Edge, 12 Star and 26 Corners (paper: 25, 30
and 28). Star files without `Standard` cannot be placed on the stimulus and are left out.

In [ ]:
FOLDS = ["P180", "S90", "S52", "S40"]
ANGLE = {"P180": 180, "S90": 90, "S52": 52, "S40": 40}
rows = []
for r, b in zip(reg, base_rows):
    for reading, base in [(k, b[col]) for k, col in BASE_COL.items()]:
        for smoothed in (True, False):
            if not smoothed and reading != "squares":
                continue
            bb = base if smoothed else b["base_squares_unsmoothed"]
            for fold, m in v.fold_measures(r, bb, smoothed=smoothed).items():
                rows.append({"file": r["file"], "cell": r["cell"], "monkey": r["monkey"], "condition": r["condition"],
                             "base_reading": reading, "smoothed": smoothed, "fold": fold, "base": bb,
                             "response": m["response"], "diff": m["diff"], "blocks": m["blocks"]})
T = pd.DataFrame(rows)
main = T[(T.base_reading == "squares") & T.smoothed]
print("Neurons per fold (main reading):", main.groupby("fold").size().to_dict())
print("Neurons per fold with a dataset BaseActivity:", T[(T.base_reading == "dataset") & T["diff"].notna()].groupby("fold").size().to_dict())
print(main.groupby("fold")[["response", "base", "diff"]].agg(["min", "median", "max"]).round(2).to_string())

def by(reading="squares", smoothed=True):
    t = T[(T.base_reading == reading) & (T.smoothed == smoothed)].dropna(subset=["diff"])
    return {f: g.set_index("file") for f, g in t.groupby("fold")}

## neuron-counts

> Of 118 V1 neurons recorded from two monkeys (102 from monkey J, 16 from monkey A), 60 were
> discarded for technical problems and 58 analysed; 7 were tested in all three conditions, 11 in
> two and 40 in one, giving 25 neurons for Square and Edge, 30 for Star and 28 for Corners.

In [ ]:
paper = {"neurons analysed": 58, "Square and Edge": 25, "Star": 30, "Corners": 28,
         "in all three": 7, "in two": 11, "in one": 40}
nconds = info.groupby("cell").condition.nunique()
ours = {"neurons analysed": nconds.size, "Square and Edge": (info.condition == "Edge-compare").sum(),
        "Star": (info.condition == "Star").sum(), "Corners": (info.condition == "Corners").sum(),
        "in all three": int((nconds == 3).sum()), "in two": int((nconds == 2).sum()), "in one": int((nconds == 1).sum())}
regi = info[info.registered]
nreg = regi.groupby("cell").condition.nunique()
ours_reg = {"neurons analysed": nreg.size, "Square and Edge": (regi.condition == "Edge-compare").sum(),
            "Star": (regi.condition == "Star").sum(), "Corners": (regi.condition == "Corners").sum(),
            "in all three": int((nreg == 3).sum()), "in two": int((nreg == 2).sum()), "in one": int((nreg == 1).sum())}
tab = pd.DataFrame({"paper": paper, "files": ours, "registered files": ours_reg})
tab["files match"] = tab.paper == tab.files
print(tab.to_string())
print("\nCells by monkey (files):", info.drop_duplicates("cell").monkey.value_counts().to_dict(),
      "; spreadsheet rows by monkey:", sheet.Cell.str[-2].value_counts().to_dict())
print("Sum over experiments, paper 7*3 + 11*2 + 40 =", 7 * 3 + 11 * 2 + 40, "; files:", len(info))

**Result: not reproduced.** The files hold 53 cells (49 from monkey J and 4 from monkey A by the
cell code), with 22 Square and Edge, 29 Star and 26 Corners files (paper 25, 30 and 28); 7 cells
are in all three experiments, as in the paper, but 10 are in two and 36 in one (paper 11 and 40).
The spreadsheet lists 58 cells, the paper's number, but five have no experiment marked and no
files. Of the 77 files, 60 can be placed on the stimulus (22, 12 and 26).

## spikes-per-condition (blocked)

> Each experimental condition required at least 10,000 spikes, and the mean number of spikes per
> condition was 35,998 (STD 25,981).

The number of spikes needs the spike count map or the eye time map, which the files do not hold:
`image` is a rate (count divided by eye time). One file of 77 also stores its eye time, and its
count is printed in the data check (one neuron and condition). The claim is blocked.

## s4-baseline-no-difference

> Absolute baseline activity did not differ significantly across the three visual stimuli
> (Square and Edge, Star, Corners).

In [ ]:
def baseline_tests(col, label):
    g = {c: B.loc[B.condition == c, col].dropna().values for c in v.CONDITIONS}
    print(f"--- baseline reading: {label}")
    print(pd.DataFrame({c: {"n": len(x), "q25": np.percentile(x, 25), "median": np.median(x), "q75": np.percentile(x, 75),
                            "min": x.min(), "max": x.max(), "mean": x.mean()} for c, x in g.items()}).round(2).to_string())
    pairs = [("Edge-compare", "Star"), ("Edge-compare", "Corners"), ("Star", "Corners")]
    p = [stats.ranksums(g[a], g[b]).pvalue for a, b in pairs]
    q = multipletests(p, method="fdr_bh")[1]
    for (a, b), pp, qq in zip(pairs, p, q):
        print(f"(a) rank sum {a} vs {b}: p = {pp:.4f}, BH = {qq:.4f}")
    kw = stats.kruskal(*g.values())
    an = stats.f_oneway(*g.values())
    print(f"(b) Kruskal-Wallis H = {kw.statistic:.2f}, p = {kw.pvalue:.4f}")
    print(f"(c) one-way ANOVA F = {an.statistic:.2f}, p = {an.pvalue:.4f}")
    return min(q), kw.pvalue, an.pvalue

res_s4 = {lab: baseline_tests(col, lab) for col, lab in [("base_squares", "squares (main)"), ("base_centred", "centred squares (variant)"),
                                                          ("base_dataset", "dataset BaseActivity")]}
for lab, (qa, pb, pc) in res_s4.items():
    print(f"{lab}: any reading significant at 0.05: {min(qa, pb, pc) < 0.05}")
# by monkey, main reading
print("\nMedian baseline by monkey and experiment (squares):")
print(B.pivot_table(index="monkey", columns="condition", values="base_squares", aggfunc=["count", "median"]).round(2).to_string())
# the 38 files that have a BaseActivity, under each reading, so that the readings are compared on the same neurons
print("\nThe same 38 files under each reading:")
B38 = B.dropna(subset=["base_dataset"])
for col in ["base_squares", "base_centred", "base_dataset"]:
    g = [B38.loc[B38.condition == c, col].values for c in v.CONDITIONS]
    print(f"{col}: medians {[round(float(np.median(x)), 2) for x in g]} (n {[len(x) for x in g]}); "
          f"Kruskal-Wallis p = {stats.kruskal(*g).pvalue:.4f}; ANOVA p = {stats.f_oneway(*g).pvalue:.4f}")


**Result: not reproduced.** With the paper's squares, the baselines differ across the stimuli
(medians: Square and Edge 7.68, Star 1.84, Corners 2.45 spikes/s), and all three readings find
it: rank sum with BH, Square and Edge against Star 0.0128 and against Corners 0.0076; Kruskal-Wallis
p = 0.0035; ANOVA p = 0.0010. With the squares centred on the corners (variant) all three readings
find it as well (BH 0.0118 and 0.0174; Kruskal-Wallis p = 0.0055; ANOVA p = 0.0019). With the
files' own `BaseActivity` (9, 10 and 19 neurons) none of the three readings is significant
(smallest BH 0.109, Kruskal-Wallis p = 0.114, ANOVA p = 0.166). On the same 38 files, the squares
still find a difference (Kruskal-Wallis p = 0.0245, centred 0.0341) where `BaseActivity` does not
(0.114), so the two baselines disagree because they measure differently, not because they cover
different neurons. The outcome rests on the baseline being the squares the Methods describe, not
the undocumented `BaseActivity`. It does not rest on where the squares are: Square and Edge
against Star alone is significant, and the readings "nearest" and "corners_quarter" also find a
difference (Kruskal-Wallis p = 0.0200 and 0.0328; section "Baseline readings compared"). Star has 12 of the paper's 30 neurons.

## fig4a-above-baseline

> For every fold (P180, S90, S52, S40), most neurons' midpoint fold responses lie above their
> absolute baseline activity, and the mass center of each fold's points lies above the line of
> unit slope, with the mass center's distance from that line increasing as the fold angle
> decreases.

In [ ]:
def fig4a(reading):
    d = by(reading)
    rows = []
    for f in FOLDS:
        t = d[f]
        pos = t[(t.response > 0) & (t.base > 0)]
        rows.append({"fold": f, "n": len(t), "above": int((t.response > t.base).sum()),
                     "frac_above": (t.response > t.base).mean(),
                     "(a) centre resp": t.response.mean(), "(a) centre base": t.base.mean(),
                     "(a) distance": t.response.mean() - t.base.mean(),
                     "(b) n log": len(pos), "(b) distance log10": np.log10(pos.response).mean() - np.log10(pos.base).mean()})
    out = pd.DataFrame(rows).set_index("fold")
    return out

for reading in ["squares", "centred", "dataset"]:
    out = fig4a(reading)
    print(f"--- baseline reading: {reading}")
    print(out.round(3).to_string())
    for col in ["(a) distance", "(b) distance log10"]:
        dd = out[col].values
        print(f"{col}: ordered P180 < S90 < S52 < S40: {bool(np.all(np.diff(dd) > 0))}; all above the line: {bool((dd > 0).all())}")
    print("more than half above, every fold:", bool((out.frac_above > 0.5).all()))

fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
d = by("squares")
for a, f in zip(ax, FOLDS):
    t = d[f]
    a.loglog(t.base.clip(lower=0.05), t.response.clip(lower=0.05), ".")
    a.plot([0.05, 200], [0.05, 200], "k:")
    a.plot(t.base.mean(), t.response.mean(), "r+", ms=12)
    a.set_title(f); a.set_xlabel("baseline (spikes/s)")
ax[0].set_ylabel("midpoint response (spikes/s)")
plt.tight_layout(); plt.show()
print("Values clipped at 0.05 spikes/s for the log axes:", int(sum(((d[f].base < 0.05) | (d[f].response < 0.05)).sum() for f in FOLDS)))

**Result: reproduced.** With the paper's squares, 20 of 22 (P180), 20 of 22 (S90), 12 of 12
(S52) and 26 of 26 (S40) neurons lie above their baseline, every mass centre lies above the unit
line, and its distance increases from P180 to S40 under both readings of the mass centre (raw
means 3.88, 8.21, 19.10, 26.41 spikes/s; log means 0.179, 0.299, 0.926, 1.036 log10 units). The
same holds with the files' `BaseActivity`. With the centred squares (variant), 18 of 22 lie above
for P180 and S90, and the order holds under reading (a) (4.27, 8.59, 20.14, 26.41) but not under
(b), where S52 (1.206) is above S40 (1.036); the criterion is met under one reading. With the Corners
squares at the quarter points the order also holds under (a) and not under (b), and with reading
"nearest" it holds under both (section "Baseline readings compared"). The Star result rests on 12
neurons.

## fig4b-spike-rate-difference

> The mean spike rate difference at the fold midpoint decreases with increasing fold angle, from
> S40 (25.60 spikes/s) through S52 (18.10) and S90 (10.69) to P180 (4.87).

In [ ]:
PAPER_MID = {"S40": (25.60, 4.18), "S52": (18.10, 2.39), "S90": (10.69, 1.94), "P180": (4.87, 1.15)}

def mid_table(reading="squares", smoothed=True):
    d = by(reading, smoothed)
    rows = []
    for f in ["S40", "S52", "S90", "P180"]:
        t = d[f]["diff"]
        exp, tol = PAPER_MID[f]
        rows.append({"fold": f, "n": len(t), "mean": t.mean(), "sem": t.sem(), "median": t.median(),
                     "paper": exp, "tolerance": tol, "within": abs(t.mean() - exp) <= tol})
    out = pd.DataFrame(rows).set_index("fold")
    out.attrs["ordered"] = bool(np.all(np.diff(out["mean"].values) < 0))
    return out

for reading, smoothed in [("squares", True), ("centred", True), ("dataset", True), ("squares", False)]:
    out = mid_table(reading, smoothed)
    print(f"--- baseline {reading}, {'smoothed' if smoothed else 'unsmoothed'} map: ordered S40 > S52 > S90 > P180: {out.attrs['ordered']}")
    print(out.round(2).to_string())
# by monkey, main reading
d = by("squares")
print("\nMean difference by monkey (main reading):")
print(pd.DataFrame({f: d[f].groupby("monkey")["diff"].agg(["count", "mean"]).stack() for f in FOLDS}).round(2).to_string())
# Corners: dark fold for reference
print(f"\nCorners dark S40 fold (not used by the paper): mean {d['S40dark']['diff'].mean():.2f} ± {d['S40dark']['diff'].sem():.2f}")

In [ ]:
# Square and Edge per neuron: midpoint responses (not baseline-subtracted) and the baselines, spikes/s
se = by("squares")
SE = pd.DataFrame({f"{f} response": se[f]["response"] for f in ["Edge", "S90", "P180"]})
SE = SE.join(B.set_index("file")[["base_squares", "base_centred", "base_dataset"]])
print(SE.round(2).to_string())
print("\nMeans:", SE.mean().round(2).to_dict())
paper_diff = {"Edge": (22.01, 3.41), "S90": (10.69, 1.94), "P180": (4.87, 1.15)}   # Table 1 midpoints ± SEM (claims table1-edge, fig4b-spike-rate-difference)
for f, (p, sem) in paper_diff.items():
    print(f"{f}: mean response {SE[f + ' response'].mean():.2f} - paper's mean difference {p} = implied mean baseline "
          f"{SE[f + ' response'].mean() - p:.2f} (squares {SE.base_squares.mean():.2f}, centred {SE.base_centred.mean():.2f})")
lo = max(SE[f + " response"].mean() - p - sem for f, (p, sem) in paper_diff.items())
hi = min(SE[f + " response"].mean() - p + sem for f, (p, sem) in paper_diff.items())
print(f"A single mean baseline puts all three means within the paper's SEM when it lies from {lo:.2f} to {hi:.2f} spikes/s "
      f"(squares {SE.base_squares.mean():.2f}, {SE.base_squares.mean() - hi:.2f} above; centred {SE.base_centred.mean():.2f}, "
      f"{SE.base_centred.mean() - hi:.2f} above)")


**Result: partially reproduced.** With the paper's squares and the smoothed map, the means are
ordered S40 > S52 > S90 > P180 (26.41, 19.10, 8.21 and 3.88 spikes/s), and S40, S52 and P180 are
within the paper's SEM, while S90 is 0.54 spikes/s below its range (8.75 to 12.63). With the
centred squares the order holds and S90 (8.59) is again the one outside. With the files'
`BaseActivity` the order holds, and S40 and S90 are within tolerance while S52 (20.57) and P180
(3.58) are not; with the unsmoothed map the order holds and P180 (3.25) is the one outside. No
variant puts all four within tolerance, and none of the readings compared before the summary does
either (S90 is outside under "nearest", 8.59, and "corners_quarter", 8.21). The Star value rests
on 12 of 30 neurons.

All three Square and Edge values are below the paper's (Edge midpoint 18.72 against 22.01, S90
8.21 against 10.69, P180 3.88 against 4.87). The per-neuron table above shows how far the baseline
accounts for this. The paper's exact means would follow from the mean responses here with a mean
baseline of 5.49 (Edge), 6.30 (S90) and 7.79 spikes/s (P180), but any mean baseline from 6.64 to
8.24 spikes/s puts all three within the paper's SEM. The squares give 8.78, 0.54 above that range,
and the centred squares 8.40, 0.16 above it. A mean baseline lower by about 0.5 to 2 spikes/s,
together with the three of the paper's 25 neurons that are missing here, is consistent with all
three gaps.

## fig4b-pairwise-significance

> The midpoint spike rate difference for S40 is significantly larger than for S90 and for P180,
> and that for S52 is significantly larger than for P180.

In [ ]:
def pairwise(reading="squares"):
    d = by(reading)
    pairs = [("S40", "S52"), ("S40", "S90"), ("S40", "P180"), ("S52", "S90"), ("S52", "P180"), ("S90", "P180")]
    rows = []
    for a, b in pairs:
        x, y = d[a]["diff"], d[b]["diff"]
        rows.append({"pair": f"{a} > {b}", "mean diff": x.mean() - y.mean(), "p": stats.ranksums(x, y).pvalue,
                     "direction ok": x.mean() > y.mean(), "named": (a, b) in [("S40", "S90"), ("S40", "P180"), ("S52", "P180")]})
    out = pd.DataFrame(rows)
    out["BH (a) six pairs"] = multipletests(out.p, method="fdr_bh")[1]
    out["BH (b) three named"] = np.nan
    out.loc[out.named, "BH (b) three named"] = multipletests(out.loc[out.named, "p"], method="fdr_bh")[1]
    return out

for reading in ["squares", "centred", "dataset"]:
    out = pairwise(reading)
    print(f"--- baseline {reading}")
    print(out.round(5).to_string(index=False))
    named = out[out.named]
    for col in ["BH (a) six pairs", "BH (b) three named"]:
        print(f"{col}: all three named pairs significant in direction: {bool(((named[col] < 0.05) & named['direction ok']).all())}")

**Result: reproduced.** With the paper's squares, S40 > S90 (BH p = 0.00012 over six pairs),
S40 > P180 (p < 0.00001) and S52 > P180 (p = 0.00005) are significant by rank sum tests with BH
correction under both families (all six pairs, or the three named). Of the pairs the text does not
claim, S52 > S90 is also significant (0.0053) and S40 > S52 and S90 > P180 are not. With the
centred squares and with the files' `BaseActivity`, the three named pairs are significant under
both families as well.

## fig5-average-maps

> The population-average spike rate difference maps show neural activity increasing as the
> corner angle decreases.

In [ ]:
def avg_map(condition, reading="squares"):
    maps = []
    for r, b in zip(reg, base_rows):
        if r["condition"] != condition:
            continue
        base = b[BASE_COL[reading]]
        if np.isfinite(base):
            maps.append((r, r["rate"] - base))
    shapes = {m.shape for _, m in maps}
    # Corners maps come in two sizes; the smaller is the larger cropped by 8 DU on each side
    H, W = max(shapes)
    stack = []
    for r, m in maps:
        pad = (H - m.shape[0]) // 2
        stack.append(np.pad(m, pad, constant_values=np.nan) if pad else m)
    ref = next(r for r, m in maps if m.shape == (H, W))
    return np.nanmean(stack, axis=0), ref, len(stack)

small = [r for r in reg if r["condition"] == "Corners" and r["rate"].shape == (257, 193)]
large = [r for r in reg if r["condition"] == "Corners" and r["rate"].shape == (273, 209)]
print("Corners fold ends (x, y at 0%, y at 100%) in the smaller maps plus 8 DU, and in the larger maps:")
for name in ["S40_up", "S40_down"]:
    print(f"  {name}: smaller + 8 {sorted({tuple(np.round(np.array(r['folds'][name]) + 8, 1)) for r in small})}; "
          f"larger {sorted({tuple(np.round(r['folds'][name], 1)) for r in large})}")

rows = []
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, cond in zip(ax, v.CONDITIONS):
    A, ref, n = avg_map(cond)
    im = a.imshow(A, cmap="viridis"); a.set_title(f"{v.CONDITIONS[cond]} (N = {n})"); plt.colorbar(im, ax=a)
    print(f"{cond}: average map range {np.nanmin(A):.2f} to {np.nanmax(A):.2f} spikes/s")
    fake = dict(ref, rate=A + 0.0, rate_unsmoothed=A + 0.0)
    for fold, m in v.fold_measures(fake, 0.0).items():
        if fold in FOLDS:
            rows.append({"fold": fold, "N": n, "midpoint of average map": m["diff"],
                         "mean of the ten blocks": np.mean(np.mean(m["blocks"], axis=0))})
plt.tight_layout(); plt.show()
f5 = pd.DataFrame(rows).set_index("fold").loc[["S40", "S52", "S90", "P180"]]
print(f5.round(2).to_string())
for col in f5.columns[1:]:
    print(f"{col}: ordered S40 > S52 > S90 > P180: {bool(np.all(np.diff(f5[col].values) < 0))}")

**Result: reproduced.** The average maps, read at the midpoint squares, are ordered S40 (26.41) >
S52 (19.07) > S90 (8.15) > P180 (3.79 spikes/s), as the criterion requires; as the claims file
notes, this repeats the means of fig4b-spike-rate-difference. The variant, the mean over the ten
fold blocks of the average map, is ordered the same way (23.14, 17.20, 7.43, 3.47). The two
Corners map sizes are concentric: the fold ends of the smaller maps plus 8 DU equal those of the
larger, as printed.

## Table 1 (table1-edge, table1-p180, table1-s90, table1-s52, table1-s40)

Ten blocks along each fold from 0% to 100% luminance, 8 DU wide, 10% of the fold long; per neuron
the minimum, maximum, mean and median of the ten block differences, and the midpoint. Reading (a):
per neuron, then mean ± SEM over neurons. Reading (b): the same summaries of the population-average
profile. For S90 and S52, reading (i) averages the two half-diagonals block by block; reading (ii)
summarises each half on its own and averages the two summaries per neuron.

In [ ]:
PAPER_T1 = {
    "Edge": [(7.98, 1.60), (33.13, 5.11), (20.44, 3.33), (20.77, 3.48), (22.01, 3.41)],
    "P180": [(0.18, 1.01), (10.28, 1.53), (4.55, 1.12), (4.03, 1.06), (4.87, 1.15)],
    "S90": [(4.36, 1.07), (14.61, 2.48), (9.22, 1.61), (9.31, 1.62), (10.69, 1.94)],
    "S52": [(6.26, 1.33), (25.39, 3.52), (15.66, 2.23), (15.93, 2.36), (18.10, 2.39)],
    "S40": [(9.33, 1.69), (35.11, 4.17), (22.81, 3.11), (22.83, 3.48), (25.60, 4.18)],
}
STATS = ["min", "max", "mean", "median", "midpoint"]

def summaries(blocks_list, half_mode):
    if half_mode == "i" or len(blocks_list) == 1:
        b = np.mean(blocks_list, axis=0)
        return np.array([np.nanmin(b), np.nanmax(b), np.nanmean(b), np.nanmedian(b)])
    return np.mean([[np.nanmin(b), np.nanmax(b), np.nanmean(b), np.nanmedian(b)] for b in blocks_list], axis=0)

def table1(fold, reading="squares"):
    t = by(reading)[fold]
    out = {}
    for half in (["i", "ii"] if fold in ("S90", "S52") else ["i"]):
        per = np.array([summaries(b, half) for b in t.blocks])
        per = np.c_[per, t["diff"].values]
        a = pd.DataFrame({"(a) mean": per.mean(0), "(a) sem": per.std(0, ddof=1) / np.sqrt(len(per))}, index=STATS)
        if half == "i":
            prof = np.mean([np.mean(b, axis=0) for b in t.blocks], axis=0)
            bvals = [np.nanmin(prof), np.nanmax(prof), np.nanmean(prof), np.nanmedian(prof), t["diff"].mean()]
        else:
            halves = np.array([np.array(b) for b in t.blocks])  # neurons x halves x blocks
            bvals = list(np.mean([[np.nanmin(p), np.nanmax(p), np.nanmean(p), np.nanmedian(p)] for p in halves.mean(0)], axis=0)) + [t["diff"].mean()]
        a["(b) population profile"] = bvals
        a["paper"] = [p for p, _ in PAPER_T1[fold]]
        a["tolerance"] = [s for _, s in PAPER_T1[fold]]
        a["(a) within"] = (a["(a) mean"] - a.paper).abs() <= a.tolerance
        a["(b) within"] = (a["(b) population profile"] - a.paper).abs() <= a.tolerance
        out[half] = a
    return out, len(t)

def report_table1(fold):
    res = {}
    for reading in ["squares", "centred", "dataset"]:
        tabs, n = table1(fold, reading)
        for half, a in tabs.items():
            key = f"{reading}, halves ({half})" if fold in ("S90", "S52") else reading
            print(f"--- {fold}, baseline {key}, N = {n}")
            print(a.round(2).to_string())
            for r in ["(a)", "(b)"]:
                k = int(a[f"{r} within"].sum())
                res[(key, r)] = k
                print(f"reading {r}: {k} of 5 within tolerance")
    return res

### table1-edge

> Table 1, Edge: minimum, maximum, mean and median over the ten blocks, and the midpoint.

In [ ]:
t1_Edge = report_table1("Edge")
d = by("squares")["Edge"]
prof = np.mean([np.mean(b, axis=0) for b in d.blocks], axis=0)
print("\nPopulation-average block profile, 0% to 100% (main reading):", np.round(prof, 2))
print("Block differences that are missing (ROI not covered):", int(sum(np.isnan(np.array(b)).sum() for b in d.blocks)))

**Result: not reproduced.** With the paper's squares (inside the corners), no reading puts three
of the five values within tolerance: reading (a) gives two (maximum 29.86 and midpoint 18.72) and
reading (b) two (median and midpoint); the minimum (5.50), mean (16.92) and median (16.78) of
reading (a) are below the paper's ranges. With the squares centred on the corners, both readings
give three of five (reading (a): maximum 30.24, mean 17.31, midpoint 19.10), which would make the
claim partially reproduced, so the outcome rests on the placement of the squares. The Methods'
wording ("at the corners of") allows both, and the centred squares are nearer the files'
`BaseActivity` for Square and Edge (data check). The Results, however, say the baseline was
calculated "inside the ROIs", which favours the squares inside the corners, and for Star
`BaseActivity` favours the inside squares. A change to partially reproduced was proposed for the
third verification round; the verifier did not agree, and the outcome stays not reproduced.
With the files' `BaseActivity` (9 neurons) no value is within tolerance.

### table1-p180

> Table 1, P180: minimum, maximum, mean and median over the ten blocks, and the midpoint.

In [ ]:
t1_P180 = report_table1("P180")
d = by("squares")["P180"]
prof = np.mean([np.mean(b, axis=0) for b in d.blocks], axis=0)
print("\nPopulation-average block profile, 0% to 100% (main reading):", np.round(prof, 2))
print("Block differences that are missing (ROI not covered):", int(sum(np.isnan(np.array(b)).sum() for b in d.blocks)))

**Result: partially reproduced.** With the paper's squares, reading (a) puts three of the five values within tolerance (maximum 10.28, mean 3.47, midpoint 3.88) and reading (b) three (mean, median, midpoint); the minimum of reading (a) is -1.71 against 0.18 ± 1.01. The centred squares give four of five under (a) (the minimum, -1.32, still outside), and the files' `BaseActivity` three of five under (a).

### table1-s90

> Table 1, S90: minimum, maximum, mean and median over the ten blocks, and the midpoint.

In [ ]:
t1_S90 = report_table1("S90")
d = by("squares")["S90"]
prof = np.mean([np.mean(b, axis=0) for b in d.blocks], axis=0)
print("\nPopulation-average block profile, 0% to 100% (main reading):", np.round(prof, 2))
print("Block differences that are missing (ROI not covered):", int(sum(np.isnan(np.array(b)).sum() for b in d.blocks)))

**Result: not reproduced.** With the paper's squares (inside the corners), no reading puts three
of the five values within tolerance: reading (a) gives one (the maximum) under either half
reading, and reading (b) none; the mean of the blocks is 7.46 against 9.22 ± 1.61. With the
squares centred on the corners, reading (a) with halves (i) gives three of five (maximum 13.80,
mean 7.84, median 7.90), which would make the claim partially reproduced, so the outcome rests on
the placement of the squares. As for table1-edge, a change to partially reproduced was proposed
for the third verification round; the verifier did not agree, and the outcome stays not
reproduced. With the files'
`BaseActivity` (9 neurons) the best reading gives two.

### table1-s52

> Table 1, S52: minimum, maximum, mean and median over the ten blocks, and the midpoint.

In [ ]:
t1_S52 = report_table1("S52")
d = by("squares")["S52"]
prof = np.mean([np.mean(b, axis=0) for b in d.blocks], axis=0)
print("\nPopulation-average block profile, 0% to 100% (main reading):", np.round(prof, 2))
print("Block differences that are missing (ROI not covered):", int(sum(np.isnan(np.array(b)).sum() for b in d.blocks)))

**Result: partially reproduced.** With the paper's squares, reading (a) puts four of the five values within tolerance under both half readings; the minimum (9.19 under halves (i), 7.61 under (ii)) is above 6.26 ± 1.33. With the centred squares the best reading gives three of five (reading (a), halves (i)), and with the files' `BaseActivity` two, so the outcome is partial under both placements of the squares and rests on the baseline being
the squares, not `BaseActivity`. The values rest on 12 of the paper's 30 Star neurons.

### table1-s40

> Table 1, S40: minimum, maximum, mean and median over the ten blocks, and the midpoint.

In [ ]:
t1_S40 = report_table1("S40")
d = by("squares")["S40"]
prof = np.mean([np.mean(b, axis=0) for b in d.blocks], axis=0)
print("\nPopulation-average block profile, 0% to 100% (main reading):", np.round(prof, 2))
print("Block differences that are missing (ROI not covered):", int(sum(np.isnan(np.array(b)).sum() for b in d.blocks)))

**Result: reproduced.** With the paper's squares and the bright S40 fold, reading (a) puts all
five values within tolerance (10.65, 34.40, 23.14, 23.15 and 26.41 spikes/s); so does the files'
`BaseActivity` (19 neurons). The centred variant does not change Corners. With the two Corners
squares at the quarter points of the bottom edge (reading "corners_quarter", section "Baseline
readings compared") only two of five are within tolerance (6.97, 30.72, 19.46, 19.47, 22.73), so
the outcome rests on the assumed placement of the Corners squares at the bottom corners of the
ROI, the placement nearer `BaseActivity` (median absolute difference 0.72 against 2.88 spikes/s).

## fig6-midpoint-representative

> For each fold and the Edge, the spike rate difference at the midpoint does not differ
> significantly from the mean or the median spike rate difference along the fold or Edge, so the
> midpoint response represents the whole fold.

In [ ]:
def fig6(reading="squares", half="i"):
    d = by(reading)
    rows = []
    for f in ["Edge", "P180", "S90", "S52", "S40"]:
        t = d[f]
        per = np.array([summaries(b, half) for b in t.blocks])
        mid = t["diff"].values
        for k, col in [("mean", 2), ("median", 3)]:
            rows.append({"fold": f, "against": k, "n": len(mid), "median(mid - along)": np.median(mid - per[:, col]),
                         "rank sum p": stats.ranksums(mid, per[:, col]).pvalue,
                         "signed rank p (variant)": stats.wilcoxon(mid, per[:, col]).pvalue})
    return pd.DataFrame(rows)

for reading in ["squares", "centred", "dataset"]:
    for half in ["i", "ii"]:
        out = fig6(reading, half)
        print(f"--- baseline {reading}, halves ({half})")
        print(out.round(4).to_string(index=False))
        print(f"rank sum tests significant at 0.05: {int((out['rank sum p'] < 0.05).sum())} of {len(out)}; "
              f"signed rank (variant): {int((out['signed rank p (variant)'] < 0.05).sum())} of {len(out)}")

**Result: reproduced.** By the paper's test (rank sum, alpha 0.05, no correction), none of the ten
comparisons of midpoint against the mean or median along the fold is significant under any of the
three baselines or either half reading (smallest p 0.36). The paired signed rank test, shown as a
variant, finds the midpoint higher than the mean and the median along S40 (p = 0.015 and 0.009) and
than the median along S90 (p = 0.025), so the absence rests on the unpaired test the paper names;
the median differences are 2.39 and 1.36 spikes/s for S40.

## fig7a-rms-contrast-fit

> Local RMS contrast along the Edge rises from 0.61 to 0.943 with luminance and is fit by an
> exponential model, f(l) = -0.34 exp(-0.05 l) + 0.95, with R2 = 0.9983.

In [ ]:
levels = np.arange(0, 100 + 1e-9, v.RMS_STEP)
edge_recs = [r for r in reg if r["condition"] == "Edge-compare"]
curves = {r["file"]: v.rms_contrast(r["lum"], r["folds"]["Edge"], levels) for r in edge_recs}
ref = curves[edge_recs[0]["file"]]
spread = max(np.nanmax(np.abs(c[["a_sd_over_mean", "b_sd_norm"]].values - ref[["a_sd_over_mean", "b_sd_norm"]].values)) for c in curves.values())
print(f"{len(curves)} Edge stimuli; largest difference between their RMS curves: {spread:.3f}")
mean_curve = pd.concat(curves.values()).groupby("level").mean()
print(mean_curve.iloc[::6].round(3).to_string())

def model(p, l):
    return p[0] * np.exp(-p[1] * l) + p[2]

PAPER_FIT = {"min": (0.61, 0.061), "max": (0.943, 0.094), "a": (-0.34, 0.034), "b": (0.05, 0.005), "c": (0.95, 0.095), "R2": (0.9983, 0.0017)}
fits = {}
for col in ["a_sd_over_mean", "b_sd_norm"]:
    y = mean_curve[col].values; l = mean_curve.index.values; ok = np.isfinite(y)
    for loss in ["linear", "soft_l1"]:   # least squares (Levenberg-Marquardt) and a robust fit standing in for LAR
        p = least_squares(lambda p: model(p, l[ok]) - y[ok], x0=[-0.34, 0.05, 0.95], loss=loss,
                          f_scale=0.01, method="lm" if loss == "linear" else "trf").x
        r2 = 1 - np.sum((y[ok] - model(p, l[ok])) ** 2) / np.sum((y[ok] - y[ok].mean()) ** 2)
        vals = {"min": np.nanmin(y), "max": np.nanmax(y), "a": p[0], "b": p[1], "c": p[2], "R2": r2}
        within = {k: abs(vals[k] - PAPER_FIT[k][0]) <= PAPER_FIT[k][1] for k in vals}
        fits[(col, loss)] = (vals, within)
        print(f"reading {col}, fit {loss}: " + ", ".join(f"{k} {vals[k]:.4f}{'' if within[k] else ' (out)'}" for k in vals))
paper_curve = model([-0.34, 0.05, 0.95], levels)
print(f"\nPaper's curve at 0, 50 and 100%: {paper_curve[0]:.3f}, {model([-0.34, 0.05, 0.95], 50):.3f}, {paper_curve[-1]:.3f}")
plt.figure(figsize=(5, 3.2))
plt.plot(mean_curve.index, mean_curve.a_sd_over_mean, ".", label="(a) SD / mean")
plt.plot(mean_curve.index, mean_curve.b_sd_norm, ".", label="(b) SD of 0-1 luminance")
plt.plot(levels, paper_curve, "k-", label="paper's fit")
plt.ylim(0, 1.5); plt.xlabel("Edge luminance (%)"); plt.ylabel("RMS contrast"); plt.legend(); plt.show()
print("Points of reading (a) above the axis (1.5):", int((mean_curve.a_sd_over_mean > 1.5).sum()))

In [ ]:
# Variant (verifier's suggestion): reading (a) on physical luminance, linear in the stimulus level,
# with the paper's 50% level of 14.07 cd/m2 (Methods, Square and edge) and an unknown black level L0
rows = []
for L0 in [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    L1 = 2 * 14.07 - L0
    cur = pd.concat([v.rms_contrast(L0 + (L1 - L0) * r["lum"] / 100, r["folds"]["Edge"], levels) for r in edge_recs]).groupby("level").mean()
    ok = cur.a_sd_over_mean.notna().values
    y = cur.a_sd_over_mean.values[ok]; l = cur.index.values[ok]
    p = least_squares(lambda p: model(p, l) - y, x0=[-0.34, 0.05, 0.95], method="lm").x
    r2 = 1 - np.sum((y - model(p, l)) ** 2) / np.sum((y - y.mean()) ** 2)
    rows.append({"L0 (cd/m2)": L0, "L1 (cd/m2)": L1, "at 0%": y[0], "at 49.5%": cur.a_sd_over_mean.loc[49.5], "at 99%": cur.a_sd_over_mean.loc[99.0],
                 "min": y.min(), "max": y.max(), "a": p[0], "b": p[1], "c": p[2], "R2": r2})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper: min 0.61, max 0.943, a -0.34, b 0.05, c 0.95, R2 0.9983")


**Result: not reproduced.** Neither reading of RMS contrast computed on the 22 Edge stimulus images
gives the paper's curve: the SD over the mean (a) is about 0.95 from 18% to 90% luminance but
rises to 2.09 at 0% and 1.40 at 99%, so its range is 0.94 to 2.09, not 0.61 to 0.943; the SD of
the 0-1 luminance (b) rises from 0.014 to 0.475. Neither fit gives the paper's parameters (a:
a = 1.16, b = 0.25, R2 = 0.89; b: a = -52.2, b = 0.0001). Reading (b) cannot exceed 0.5 for any
image. In the variant on physical luminance (14.07 cd/m2 at 50%, black level 0 to 10 cd/m2), no
black level gives the paper's curve either: a black level of 0.5 cd/m2 starts near the paper's
0.61 (0.55) but rises to 1.33, and the levels that keep the curve below 1 start below 0.05. The
definition of RMS contrast the paper used (its reference [21]) is not in the text read, and the
question whether the claims file's readings are the ones the paper allows is put to a reader.

## fig7c-fold-edge-equivalence

> Neural responses to P180 are roughly equivalent to those to a 50% contrast Edge, while
> responses to S40 are close to those to a 90% contrast Edge.

In [ ]:
A_, B_, C_ = -0.34, 0.05, 0.95
rms_levels = np.round(np.arange(0.610, 0.943 + 1e-9, 0.037), 3)
lum_at_rms = -np.log((rms_levels - C_) / A_) / B_
print("RMS levels:", rms_levels)
print("Luminance (%) of each level by the inverse of the paper's fit:", np.round(lum_at_rms, 1))

def edge_table(reading="squares"):
    rows = []
    for r, b in zip(reg, base_rows):
        if r["condition"] != "Edge-compare":
            continue
        base = b[BASE_COL[reading]]
        if not np.isfinite(base):
            continue
        rows.append({"file": r["file"],
                     **{("lum", L): v.edge_response(r["rate"], r["folds"]["Edge"], L) - base for L in range(0, 101, 10)},
                     **{("rms", c): v.edge_response(r["rate"], r["folds"]["Edge"], L) - base for c, L in zip(rms_levels, lum_at_rms)}})
    return pd.DataFrame(rows).set_index("file")

def equivalence(reading="squares"):
    E = edge_table(reading)
    m = mid_table(reading)
    out = {}
    for kind in ["lum", "rms"]:
        cols = [c for c in E.columns if c[0] == kind]
        means = E[cols].mean()
        lv = np.array([c[1] for c in cols], float)
        near = {f: lv[np.argmin(np.abs(means.values - m.loc[f, "mean"]))] for f in ["P180", "S40"]}
        out[kind] = (means, near)
        print(f"--- baseline {reading}, Edge levels by {'luminance %' if kind == 'lum' else 'RMS contrast'}:")
        print("Edge mean response:", " ".join(f"{l:g}:{x:.2f}" for l, x in zip(lv, means.values)))
        print(f"P180 mean {m.loc['P180', 'mean']:.2f} -> nearest level {near['P180']:g}; S40 mean {m.loc['S40', 'mean']:.2f} -> nearest level {near['S40']:g}")
    ok_a = abs(out["lum"][1]["P180"] - 50) <= 10 and abs(out["lum"][1]["S40"] - 90) <= 10
    step = 0.037
    ok_b = out["rms"][1]["P180"] == rms_levels[0] and abs(out["rms"][1]["S40"] - 0.90) <= step + 1e-9
    print(f"(a) within one step of 50% and 90%: {ok_a}; (b) P180 at the lowest level and S40 within one step of 0.90: {ok_b}")
    return E

E_main = equivalence("squares")
_ = equivalence("centred")
_ = equivalence("dataset")

In [ ]:
# Variants of the rule. "One step" read on the grid: the nearest level is at most one grid step
# from the level nearest 0.90 (0.906), instead of within 0.037 of 0.90 itself
def grid_step(reading):
    E = edge_table(reading); m = mid_table(reading)
    means = E[[("rms", c) for c in rms_levels]].mean().values
    near = {f: int(np.argmin(np.abs(means - m.loc[f, "mean"]))) for f in ["P180", "S40"]}
    i90 = int(np.argmin(np.abs(rms_levels - 0.90)))
    below = m.loc["P180", "mean"] < means.min(); above = m.loc["S40", "mean"] > means.max()
    print(f"{reading}: P180 nearest {rms_levels[near['P180']]}, S40 nearest {rms_levels[near['S40']]}, level nearest 0.90 is "
          f"{rms_levels[i90]}; S40 within one grid step: {abs(near['S40'] - i90) <= 1}; "
          f"P180 below every Edge level: {below}; S40 above every Edge level: {above}")
for reading in ["squares", "centred", "dataset"]:
    grid_step(reading)

# Variant named in the criterion: RMS levels from the refit of fig7a-rms-contrast-fit
for (col, loss), (vals, _) in fits.items():
    if loss != "linear":
        continue
    with np.errstate(invalid="ignore", divide="ignore"):
        lum_refit = -np.log((rms_levels - vals["c"]) / vals["a"]) / vals["b"]
    inside = np.isfinite(lum_refit) & (lum_refit >= 0) & (lum_refit <= 100)
    print(f"refit {col}: luminance of the ten levels {np.round(lum_refit, 1)}; levels inside 0-100%: {int(inside.sum())} of 10")


**Result: not reproduced.** With the paper's squares, the P180 mean (3.88 spikes/s) is below the
Edge response at every level (8.61 at 0% luminance and 17.19 at 50%), so under reading (a) it is
nearest the 0% level, not 50%, while S40 (26.41) is nearest 90%, as the paper says. Under reading
(b), P180 is at the lowest level, as the rule allows, but S40 is nearest 0.943, 0.043 from 0.90,
more than one step (0.037). Read on the grid of levels (within one level of 0.906, the level
nearest 0.90), S40 at 0.943 passes, and reading (b) would be met; but under (b) P180 lies below
and S40 above every Edge level, so "nearest" picks the two ends of the range whatever the
responses. The outcome rests on reading "one step" as 0.037 from 0.90, the criterion's wording.
The centred squares and the files' `BaseActivity` give the same nearest levels under (b), and
under (a) P180 nearest 0% (both) and S40 nearest 90% (centred) or 80% (`BaseActivity`). The refit
of fig7a-rms-contrast-fit cannot place the levels: under reading (a) its inverse is undefined at
every level, under (b) every level falls above 100% luminance.

## fig7c-fold-edge-significance

> Responses to P180 are significantly lower than responses to a 94% contrast Edge, and responses
> to S40 are significantly higher than responses to an Edge of less than 68% contrast.

In [ ]:
def fold_edge_tests(reading="squares"):
    E = edge_table(reading)
    d = by(reading)
    rows = []
    for f in FOLDS:
        x = d[f]["diff"].values
        for c in rms_levels:
            y = E[("rms", c)].dropna().values
            t = stats.ttest_ind(x, y)
            rows.append({"fold": f, "rms": c, "fold - edge": x.mean() - y.mean(), "t": t.statistic, "p": t.pvalue})
    T7 = pd.DataFrame(rows)
    T7["BH all 40"] = multipletests(T7.p, method="fdr_bh")[1]
    T7["BH per fold"] = T7.groupby("fold").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
    T7["Bonf all 40"] = np.minimum(T7.p * len(T7), 1)
    T7["Bonf per fold"] = np.minimum(T7.p * 10, 1)
    return T7

for reading in ["squares", "centred", "dataset"]:
    T7 = fold_edge_tests(reading)
    print(f"--- baseline {reading}")
    print(T7[T7.fold.isin(["P180", "S40"])].round(4).to_string(index=False))
    for col in ["p", "BH all 40", "BH per fold", "Bonf all 40", "Bonf per fold"]:
        p180 = T7[(T7.fold == "P180") & (T7.rms == rms_levels[-1])].iloc[0]
        s40 = T7[(T7.fold == "S40") & (T7.rms.isin(rms_levels[:2]))]
        ok = (p180[col] < 0.05 and p180["fold - edge"] < 0) and bool(((s40[col] < 0.05) & (s40["fold - edge"] > 0)).all())
        sig = T7[(T7.fold == "S40") & (T7[col] < 0.05) & (T7["fold - edge"] > 0)].rms.tolist()
        print(f"{col}: claim holds {ok}; S40 significantly higher at RMS {sig}")

**Result: reproduced.** With the paper's squares, P180 is significantly lower than the Edge at
RMS 0.943 (p = 0.0005) and S40 significantly higher than the Edge at 0.610 and 0.647 (p = 0.0007
and 0.0009), under every correction and family. With Bonferroni over the 40 tests, S40 is
significant at 0.610 and 0.647 only, the levels below 0.68; with BH over 40 it is significant up to
0.795. The centred squares give the same under every correction. With the files' `BaseActivity`,
the claim holds uncorrected and with BH, not with Bonferroni.

## fig7d-nonlinear

> The relationship between corner angle and V1 spike rate difference is parametric but
> apparently not linear.

In [ ]:
for reading in ["squares", "centred", "dataset"]:
    m = mid_table(reading).loc[["S40", "S52", "S90", "P180"]]
    ang = np.array([40, 52, 90, 180], float)
    slope, icpt = np.polyfit(ang, m["mean"].values, 1)
    resid = m["mean"].values - (slope * ang + icpt)
    off = np.abs(resid) > m["sem"].values
    print(f"--- baseline {reading}: line {slope:.4f} * angle + {icpt:.2f}")
    print(pd.DataFrame({"angle": ang, "mean": m["mean"].values, "sem": m["sem"].values, "residual": resid, "off by > SEM": off}, index=m.index).round(3).to_string())
    print(f"monotonic: {m.attrs['ordered']}; non-linear by the rule (a point off the line by more than its SEM): {bool(off.any())}")
    seg = np.diff(m["mean"].values) / np.diff(ang)
    seg_se = np.sqrt(m["sem"].values[1:] ** 2 + m["sem"].values[:-1] ** 2) / np.diff(ang)
    print("variant, slopes 40-52, 52-90, 90-180:", np.round(seg, 4), "SE:", np.round(seg_se, 4))
    for i in range(2):
        diff = abs(seg[i] - seg[i + 1]); se = np.hypot(seg_se[i], seg_se[i + 1])
        print(f"  slope {i + 1} vs {i + 2}: difference {diff:.4f}, propagated SE {se:.4f}, differ by more: {diff > se}")

**Result: reproduced.** The four means are monotonic in angle, and S40, S90 and P180 lie off the
least-squares line by more than their SEM (residuals 4.77, -6.26 and 2.31 spikes/s), so the relation
is judged non-linear under the criterion's rule. In the variant, the slope from 52 to 90 degrees
(-0.287) differs from the slope from 90 to 180 (-0.048) by more than their propagated SE (0.098);
the first two slopes do not differ by more than theirs. The centred squares and the files'
`BaseActivity` give the same judgement.

## fig7d-dog-similar and s3-latency-peak (blocked)

fig7d-dog-similar needs the output of the difference-of-Gaussians model of the paper's reference
[3], which is not in the data set and is not published as data. s3-latency-peak needs maps made at
latencies other than 60 ms, which need spike times and eye positions; the data set holds only maps.

## Baseline readings compared

Every claim that uses the absolute baseline, under each reading of it: the main reading
("squares"), the squares centred on the corners ("centred"), the placement nearer the files'
`BaseActivity` per stimulus ("nearest", see the data check), the Corners squares moved along the
bottom edge of the ROI ("corners_quarter") and the files' `BaseActivity` ("dataset").

In [ ]:
# Every claim that uses a baseline, under each reading of it, including those added in round 2: "nearest" (per
# stimulus, the placement of the squares nearer the files' BaseActivity) and "corners_quarter" (another placement of
# the two Corners squares on 0% luminance)
import contextlib
import io

def decide(reading):
    out = {}
    col = BASE_COL[reading]
    g = {c: B.loc[B.condition == c, col].dropna().values for c in v.CONDITIONS}
    pairs = [("Edge-compare", "Star"), ("Edge-compare", "Corners"), ("Star", "Corners")]
    q = multipletests([stats.ranksums(g[a], g[b]).pvalue for a, b in pairs], method="fdr_bh")[1]
    out["s4: smallest BH, Kruskal-Wallis p, ANOVA p"] = (f"{min(q):.4f}, {stats.kruskal(*g.values()).pvalue:.4f}, "
                                                          f"{stats.f_oneway(*g.values()).pvalue:.4f}")
    a4 = fig4a(reading)
    out["fig4a: more than half above; ordered (a); ordered (b)"] = (
        f"{bool((a4.frac_above > 0.5).all())}; {bool(np.all(np.diff(a4['(a) distance'].values) > 0))}; "
        f"{bool(np.all(np.diff(a4['(b) distance log10'].values) > 0))}")
    m = mid_table(reading)
    out["fig4b: means S40, S52, S90, P180 (within)"] = ", ".join(f"{m.loc[f, 'mean']:.2f}{'' if m.loc[f, 'within'] else ' (out)'}"
                                                                 for f in m.index) + f"; ordered {m.attrs['ordered']}"
    pw = pairwise(reading)
    out["fig4b pairwise: largest BH of the named pairs"] = f"{pw.loc[pw.pair.isin(['S40 > S90', 'S40 > P180', 'S52 > P180']), 'BH (a) six pairs'].max():.5f}"
    for fold in ["Edge", "P180", "S90", "S52", "S40"]:
        tabs, n = table1(fold, reading)
        best = max(int(a[f"{r} within"].sum()) for a in tabs.values() for r in ["(a)", "(b)"])
        a = tabs["i"]
        out[f"table1-{fold.lower()}: best of 5 (reading (a), halves (i))"] = (
            f"{best} ({int(a['(a) within'].sum())}: " + ", ".join(f"{x:.2f}" for x in a["(a) mean"]) + f"), N = {n}")
    out["fig6: smallest rank sum p, either half reading"] = f"{min(fig6(reading, h)['rank sum p'].min() for h in ['i', 'ii']):.3f}"
    mids = {}
    for cond in v.CONDITIONS:
        A, ref, n = avg_map(cond, reading)
        for fold, mm in v.fold_measures(dict(ref, rate=A + 0.0, rate_unsmoothed=A + 0.0), 0.0).items():
            if fold in FOLDS:
                mids[fold] = mm["diff"]
    mids = [mids[f] for f in ["S40", "S52", "S90", "P180"]]
    out["fig5: average-map midpoints S40, S52, S90, P180"] = ", ".join(f"{x:.2f}" for x in mids) + f"; ordered {bool(np.all(np.diff(mids) < 0))}"
    with contextlib.redirect_stdout(io.StringIO()) as buf:
        equivalence(reading)
    out["fig7c-equivalence: (a); (b)"] = buf.getvalue().strip().splitlines()[-1]
    T7 = fold_edge_tests(reading)
    p180 = T7[(T7.fold == "P180") & (T7.rms == rms_levels[-1])].iloc[0]
    s40 = T7[(T7.fold == "S40") & (T7.rms.isin(rms_levels[:2]))]
    out["fig7c-significance: holds under p, BH 40, Bonferroni 40"] = ", ".join(
        str((p180[c] < 0.05 and p180["fold - edge"] < 0) and bool(((s40[c] < 0.05) & (s40["fold - edge"] > 0)).all()))
        for c in ["p", "BH all 40", "Bonf all 40"])
    md = m.loc[["S40", "S52", "S90", "P180"]]
    ang = np.array([40, 52, 90, 180], float)
    resid = md["mean"].values - np.polyval(np.polyfit(ang, md["mean"].values, 1), ang)
    out["fig7d: points off the line by more than SEM"] = ", ".join(f for f, o in zip(md.index, np.abs(resid) > md["sem"].values) if o)
    return out

for reading in ["squares", "centred", "nearest", "corners_quarter", "dataset"]:
    print(f"--- baseline {reading}")
    for k, val in decide(reading).items():
        print(f"  {k}: {val}")


**Result.** Under reading "nearest" every outcome is the one of the main reading except
table1-edge and table1-s90, which reach three of five under reading (a) (Edge: maximum 30.24, mean
17.31, midpoint 19.10; S90: maximum 13.80, mean 7.84, median 7.90), as with the centred squares,
and so would be partially reproduced. The other claims under "nearest": the baselines still
differ (smallest BH 0.0348, Kruskal-Wallis p = 0.0200, ANOVA p = 0.0047); fig4a holds under both
mass-centre readings; the fig4b means are ordered with S90 (8.59) outside its tolerance; the
named pairs are significant (largest BH 0.00022); table1-p180 gives four of five, table1-s52 four
and table1-s40 five; the smallest fig6 rank sum p is 0.419; the average maps are ordered; neither
reading of fig7c-fold-edge-equivalence is met; fig7c-fold-edge-significance holds under every
correction; S40, S90 and P180 lie off the fig7d line.

Under "corners_quarter", table1-s40 falls to two of five (minimum 6.97, maximum 30.72, mean
19.46, median 19.47, midpoint 22.73), which would make it not reproduced, so its outcome rests on
the Corners placement. Every other outcome is the same as under the main reading: the baselines
differ (Kruskal-Wallis p = 0.0328), fig4a holds under reading (a), the fig4b means are ordered
with S90 outside (S40 22.73 within), the named pairs are significant (largest BH 0.00288),
fig7c-fold-edge-significance holds uncorrected and with BH, and S90 and P180 lie off the fig7d
line.

Proposed and not agreed: table1-edge and table1-s90 from not reproduced to partially
reproduced, taking the centred squares for Square and Edge. The third verification round kept
both as not reproduced: the Results say the baseline was calculated "inside the ROIs", which
favours the squares inside the corners, `BaseActivity` favours the inside squares for Star, and
`BaseActivity` itself gives at most two of five for both claims. The outcomes stay as verified.

## Summary

In [ ]:
print(f"Total run time {time.time() - started:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

| Claim | Outcome |
|---|---|
| neuron-counts | not reproduced |
| spikes-per-condition | blocked (data missing) |
| s4-baseline-no-difference | not reproduced (assumption: the baseline is the Methods' squares, not `BaseActivity`) |
| fig4a-above-baseline | reproduced |
| fig4b-spike-rate-difference | partially reproduced |
| fig4b-pairwise-significance | reproduced |
| fig5-average-maps | reproduced (assumption: the measure of the maps) |
| table1-edge | not reproduced (assumption: squares inside the corners; partially reproduced with the centred squares, proposed and not agreed) |
| table1-p180 | partially reproduced |
| table1-s90 | not reproduced (assumption: squares inside the corners; partially reproduced with the centred squares, proposed and not agreed) |
| table1-s52 | partially reproduced (assumption: the baseline is the Methods' squares, not `BaseActivity`) |
| table1-s40 | reproduced (assumption: Corners squares at the bottom corners of the ROI) |
| fig6-midpoint-representative | reproduced |
| fig7a-rms-contrast-fit | not reproduced |
| fig7c-fold-edge-equivalence | not reproduced (assumption: one step is 0.037 from 0.90) |
| fig7c-fold-edge-significance | reproduced |
| fig7d-nonlinear | reproduced (assumption: the rule for non-linearity) |
| fig7d-dog-similar | blocked (data missing) |
| s3-latency-peak | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: it uses the maps the files hold, registered by the
authors (`Standard`), and so only 60 of the 77 neuron and condition maps (Star 12 of 29); the
paper used 83. The 16 Star maps that cannot be registered do not differ detectably from the 12
that can in their gaze-frame rates. The baseline squares follow the Methods for Square and Edge and
Star, inside the corners of the circumscribing square; squares centred on the corners are run as a
variant, and they turn table1-edge and table1-s90 into partial reproductions. The centred squares
are nearer the files' `BaseActivity` for Square and Edge, and that change was proposed; the third
verification round did not agree, since the Results place the baseline "inside the ROIs" and
`BaseActivity` favours the inside squares for Star. The Corners squares are taken at the bottom corners of the ROI, the placement
nearer `BaseActivity`; at the quarter points of the bottom edge table1-s40 would not be
reproduced. The files' undocumented `BaseActivity`, which the files use to normalise
`smoothed_image_norm`, gives no baseline difference across stimuli where the squares do. The Edge
contrast levels use the paper's fitted parameters, since no reading of RMS contrast reproduces the
fit.

What was found about the data set: the data set holds 53 cells and 77 maps, not the paper's 58
neurons and 83 maps; 17 Star maps lack the registration to the stimulus, which the files do not
otherwise allow; the files hold undocumented fields (`Standard`, `BaseActivity`, `norm`, always 0,
`eyetime`, `original_image`), and `smoothed_image_norm` is the smoothed map minus `BaseActivity` in 5
files, the index (map - b) / (map + b) in 12, and neither in 21; `image` is a rate, not the count the descriptor names; the stored
smoothing differs from the filter the paper describes (SD 7 DU); the spike counts and eye times
needed for the spike totals are missing; two files have a `BaseActivity` of exactly 0; the
spreadsheet lists 58 cells, five of them with no data.